## Hướng Dẫn Chuyển File Google Drive Giữa Các Tài Khoản (Nhanh & Dễ - 2025)

Hướng dẫn từng bước cách chuyển file Google Drive từ tài khoản này sang tài khoản khác bằng Google Colab! Tự động hóa quy trình mà không cần tải xuống hay tải lên thủ công. Phù hợp cho người mới, sinh viên và chuyên gia quản lý nhiều tài khoản Drive.

In [ ]:
#@title 🔐 Quản Lý Quyền Truy Cập Google Drive
#@markdown ---
#@markdown ## ⚡ Phần 1: Xác Thực & Khám Phá
#@markdown Chạy cell để kết nối tài khoản Google của bạn.
#@markdown Danh sách **Shared Drive** sẽ tự động hiển thị bên dưới để tiện sao chép tên.

# --- Cài Đặt & Xác Thực ---
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib -q
from google.colab import auth
from googleapiclient.discovery import build
import google.auth
from googleapiclient.http import BatchHttpRequest
import time
from tqdm.notebook import tqdm
import os

# Xác thực người dùng
auth.authenticate_user()
# Lấy thông tin xác thực và tạo dịch vụ Drive
# Get default credentials and build the Drive service
creds, _ = google.auth.default()
drive_service = build('drive', 'v3', credentials=creds)

print("✅ Xác thực thành công.")
# --- Liệt kê Shared Drive cho tiện ---
# --- List Shared Drives for user convenience ---
try:
    print("\n--- Danh Sách Shared Drive ---")
    drives_response = drive_service.drives().list().execute()
    shared_drives = drives_response.get('drives', [])
    if not shared_drives:
        print("Không tìm thấy Shared Drive nào.")
    else:
        for drive in shared_drives:
            print(f"- Tên: '{drive.get('name')}' (ID: {drive.get('id')})")
    print("-----------------------------\n")
except Exception as e:
    print(f"⚠️ Không thể liệt kê Shared Drive: {e}")


#@markdown ---
#@markdown ## 📝 Phần 2: Cấu Hình & Thực Thi
#@markdown Điền thông tin bên dưới rồi chạy cell.

#@markdown ---
#@markdown ### 🎬 Hành Động
#@markdown Chọn **Share** để cấp quyền hoặc **Unshare** để gỡ quyền truy cập.
action = "Share" #@param ["Share", "Unshare"]

#@markdown ---
#@markdown ### 📂 Nguồn — Drive & Đường Dẫn
#@markdown **Shared Drive** — Nhập tên chính xác. Để trống nếu dùng **My Drive**.
shared_drive_name = "" #@param {type:"string"}
#@markdown **Đường dẫn** — Đường dẫn tới file hoặc thư mục (tương đối với drive đã chọn).
#@markdown > *Ví dụ thư mục:* `/My Project Folder` — *Ví dụ file:* `/My Project Folder/document.docx`
source_path = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🎯 Người Nhận
#@markdown Nhập email của người dùng cần quản lý quyền.
destination_email = "" #@param {type:"string"}

#@markdown ---
#@markdown ### ⚙️ Tùy Chọn Nâng Cao

#@markdown **Phạm vi** — Áp dụng cho nội dung bên trong (nếu nguồn là thư mục).
sharing_scope = "Files and Folders" #@param ["Files and Folders", "Files Only", "Folders Only"]

#@markdown **Vai trò** — Mức quyền cấp cho người nhận (chỉ dùng cho Share).
sharing_role = "writer" #@param ["writer", "commenter", "reader"]

#@markdown **Gửi email thông báo** — Người nhận sẽ nhận email khi được chia sẻ.
send_notification = False #@param {type:"boolean"}


def get_item_id_from_path(drive_id, is_shared_drive, path):
    """
    Translates a path into a Google Drive file/folder ID and its mimeType,
    supporting both My Drive and Shared Drives.
    """
    clean_path = path.strip().strip('/')
    path_components = clean_path.split('/') if clean_path else []

    current_id = drive_id
    item_info = None

    for component in path_components:
        if not component: continue
        safe_name = component.replace("'", "\\'")
        query = f"name='{safe_name}' and '{current_id}' in parents and trashed=false"
        try:
            list_kwargs = dict(
                q=query, fields='files(id, name, mimeType)', pageSize=2,
                supportsAllDrives=True, includeItemsFromAllDrives=True)
            if is_shared_drive:
                list_kwargs['driveId'] = drive_id
                list_kwargs['corpora'] = 'drive'
            else:
                list_kwargs['corpora'] = 'user'
            response = drive_service.files().list(**list_kwargs).execute()
            files = response.get('files', [])
            if not files:
                print(f"❌ LỖI: Không tìm thấy thành phần đường dẫn '{component}'.")
                return None, None
            if len(files) > 1: print(f"⚠️ Cảnh báo: Tìm thấy nhiều mục tên '{component}'. Sử dụng mục đầu tiên.")
            item_info = files[0]
            current_id = item_info['id']
        except Exception as e:
            print(f"❌ Lỗi khi phân giải thành phần đường dẫn '{component}': {e}")
            return None, None

    if not path_components:
        item_info = drive_service.files().get(fileId=drive_id, fields='id, name, mimeType', supportsAllDrives=True).execute()

    return item_info.get('id'), item_info.get('mimeType')

def manage_permissions_by_path():
    """
    Finds the item at the specified path and shares or unshares it (or its contents).
    """
    # --- Step 1: Determine the target drive ---
    target_drive_id = 'root'
    is_shared_drive = False
    drive_display_name = "'My Drive'"

    if shared_drive_name:
        print(f"Đang tìm Shared Drive: '{shared_drive_name}'...")
        try:
            safe_drive_name = shared_drive_name.replace("'", "\\'")
            drive_response = drive_service.drives().list(q=f"name = '{safe_drive_name}'").execute()
            drives = drive_response.get('drives', [])
            if not drives:
                print(f"❌ LỖI: Không tìm thấy Shared Drive '{shared_drive_name}'.")
                return
            target_drive_id = drives[0]['id']
            drive_display_name = f"Shared Drive '{shared_drive_name}'"
            is_shared_drive = True
            print(f"✅ Đã tìm thấy {drive_display_name} (ID: {target_drive_id})")
        except Exception as e:
            print(f"❌ Lỗi khi tìm Shared Drive: {e}")
            return

    # --- Step 2: Resolve the path to an item ID ---
    print(f"\nĐang phân giải đường dẫn: '{source_path}' trong {drive_display_name}...")
    item_id, mime_type = get_item_id_from_path(target_drive_id, is_shared_drive, source_path)
    if not item_id: return

    # --- Step 3: Get list of items to process based on path and scope ---
    items_to_process = []
    if mime_type == 'application/vnd.google-apps.folder':
        print(f"✅ Đường dẫn trỏ tới thư mục. Áp dụng phạm vi: '{sharing_scope}'.")
        base_query = f"'{item_id}' in parents and trashed=false"
        scope_query = ""
        if sharing_scope == "Files Only": scope_query = " and mimeType != 'application/vnd.google-apps.folder'"
        elif sharing_scope == "Folders Only": scope_query = " and mimeType = 'application/vnd.google-apps.folder'"
        page_token = None
        while True:
            try:
                list_kwargs = dict(
                    q=base_query + scope_query, fields="nextPageToken, files(id, name)", pageSize=1000,
                    supportsAllDrives=True, includeItemsFromAllDrives=True, pageToken=page_token)
                if is_shared_drive:
                    list_kwargs['driveId'] = target_drive_id
                    list_kwargs['corpora'] = 'drive'
                else:
                    list_kwargs['corpora'] = 'user'
                response = drive_service.files().list(**list_kwargs).execute()
                items_to_process.extend(response.get('files', []))
                page_token = response.get('nextPageToken', None)
                if page_token is None: break
            except Exception as e:
                print(f"❌ Lỗi khi liệt kê nội dung thư mục: {e}")
                return
    else:
        print("✅ Đường dẫn trỏ tới một file đơn.")
        item_name_req = drive_service.files().get(fileId=item_id, fields='name', supportsAllDrives=True).execute()
        items_to_process.append({'id': item_id, 'name': item_name_req.get('name', 'Unknown File')})

    if not items_to_process:
        print("📁 Không tìm thấy mục nào để xử lý theo đường dẫn và phạm vi đã chọn.")
        return

    # --- Step 4: Execute the chosen action ---
    print(f"\nChuẩn bị {action.upper()} {len(items_to_process)} mục cho '{destination_email}'...")
    batch_size_limit = 100

    if action == "Share":
        permission = {'type': 'user', 'role': sharing_role, 'emailAddress': destination_email}
        share_errors = []
        def share_callback(request_id, response, exception):
            if exception:
                share_errors.append(str(exception))
        print("Đang thực thi yêu cầu chia sẻ hàng loạt...")
        for i in range(0, len(items_to_process), batch_size_limit):
            chunk = items_to_process[i:i + batch_size_limit]
            batch = drive_service.new_batch_http_request()
            for item in chunk:
                batch.add(drive_service.permissions().create(
                    fileId=item['id'], body=permission,
                    sendNotificationEmail=send_notification,
                    supportsAllDrives=True), callback=share_callback)
            try:
                batch.execute()
            except Exception as e:
                print(f"❌ Chia sẻ hàng loạt thất bại (chunk {i//batch_size_limit + 1}): {e}")
        if share_errors:
            print(f"⚠️ {len(share_errors)} lỗi khi chia sẻ:")
            for err in share_errors[:5]:
                print(f"  - {err}")
            if len(share_errors) > 5:
                print(f"  ... và {len(share_errors) - 5} lỗi khác.")

    elif action == "Unshare":
        permissions_to_delete = []
        print("Đang tìm quyền hiện có để gỡ...")
        for item in tqdm(items_to_process, desc="Đang kiểm tra quyền"):
            try:
                perms = drive_service.permissions().list(fileId=item['id'], fields='permissions(id, emailAddress)', supportsAllDrives=True).execute()
                for p in perms.get('permissions', []):
                    if p.get('emailAddress', '').lower() == destination_email.lower():
                        permissions_to_delete.append({'fileId': item['id'], 'permissionId': p['id']})
                        break
            except Exception as e:
                print(f"\n⚠️ Không thể kiểm tra quyền cho '{item['name']}'. Bỏ qua. Lý do: {e}")

        if not permissions_to_delete:
            print(f"Không tìm thấy quyền nào của '{destination_email}' trên các mục đã chọn.")
            return

        print(f"\nTìm thấy {len(permissions_to_delete)} quyền cần gỡ. Đang thực thi xóa hàng loạt...")
        unshare_errors = []
        def unshare_callback(request_id, response, exception):
            if exception:
                unshare_errors.append(str(exception))
        for i in range(0, len(permissions_to_delete), batch_size_limit):
            chunk = permissions_to_delete[i:i + batch_size_limit]
            batch = drive_service.new_batch_http_request()
            for perm in chunk:
                batch.add(drive_service.permissions().delete(fileId=perm['fileId'], permissionId=perm['permissionId'], supportsAllDrives=True), callback=unshare_callback)
            try:
                batch.execute()
            except Exception as e:
                print(f"❌ Gỡ chia sẻ hàng loạt thất bại (chunk {i//batch_size_limit + 1}): {e}")
        if unshare_errors:
            print(f"⚠️ {len(unshare_errors)} lỗi khi gỡ chia sẻ:")
            for err in unshare_errors[:5]:
                print(f"  - {err}")
            if len(unshare_errors) > 5:
                print(f"  ... và {len(unshare_errors) - 5} lỗi khác.")

    print(f"\n✅ Đã xử lý xong tất cả {len(items_to_process)} mục.")

# --- Execute the Main Function ---
if destination_email and "@" in destination_email:
    manage_permissions_by_path()
else:
    print("Vui lòng nhập địa chỉ email người nhận hợp lệ trước khi chạy.")


In [ ]:
#@title 📋 Sao Chép 'Được Chia Sẻ Với Tôi' Sang My Drive
#@markdown ---
#@markdown ## ⚡ Phần 1: Xác Thực & Thiết Lập
#@markdown Cài đặt thư viện, kết nối tài khoản Google, và thiết lập dịch vụ Drive
#@markdown với timeout dài cho các thao tác lớn.

# --- Cài Đặt & Xác Thực ---
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib -q
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import BatchHttpRequest, MediaIoBaseUpload, MediaIoBaseDownload
import google.auth
from google.auth.transport.requests import Request
import google_auth_httplib2
import httplib2
import time
import math
import json
import io
from datetime import datetime, timezone
from tqdm.notebook import tqdm

# --- Xác Thực ---
print("Đang xác thực người dùng...")
auth.authenticate_user()

# --- Thiết Lập Dịch Vụ Drive Với Timeout Tăng ---
print("Đang thiết lập dịch vụ Google Drive...")
creds, _ = google.auth.default()
creds.refresh(Request())
# Sử dụng httplib2 với timeout dài (600s = 10 phút) cho tất cả API calls
http = httplib2.Http(timeout=600)
authorized_http = google_auth_httplib2.AuthorizedHttp(creds, http=http)
drive_service = build('drive', 'v3', http=authorized_http)

# Lấy email tài khoản hiện tại
try:
    about = drive_service.about().get(fields='user(emailAddress)').execute()
    current_account = about['user']['emailAddress']
except Exception:
    current_account = "unknown"
print(f"✅ Xác thực thành công với tài khoản: {current_account}")

#@markdown ---
#@markdown ## 📝 Phần 2: Cấu Hình & Chạy Quá Trình Sao Chép
#@markdown Điền form bên dưới, sau đó chạy cell để bắt đầu.

#@markdown ---
#@markdown ### 📂 Nguồn — Mục Cần Sao Chép
#@markdown Nhập tên **chính xác** của mục cần copy từ danh sách "Được chia sẻ với tôi".
#@markdown > *Để trống để xử lý **TẤT CẢ** mục trong "Được chia sẻ với tôi".*
specific_item_name = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🎯 Đích — Thư Mục Lưu Trữ
#@markdown Nhập tên thư mục trong **My Drive** để sao chép vào.
#@markdown > *Để trống để sao chép vào thư mục gốc My Drive.*
destination_folder_name = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 📋 Phạm Vi & Hành Vi

#@markdown **Loại nội dung** — Chọn file, thư mục, hoặc cả hai.
copy_scope = "Files and Folders" #@param ["Files and Folders", "Files Only", "Folders Only"]

#@markdown **Bỏ qua mục đã tồn tại** — Tự động skip file/thư mục trùng tên ở đích.
skip_existing_items = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### ⚙️ Cài Đặt Nâng Cao

#@markdown **Giới hạn quota (GB)** — Google Drive giới hạn copy 750GB/ngày. Đặt `0` để tắt.
daily_copy_limit_gb = 700 #@param {type:"integer"}

#@markdown **Số lần thử lại** — Khi batch thất bại, sẽ thử lại tối đa bao nhiêu lần.
max_retries_per_chunk = 5 #@param {type:"integer"}

#@markdown **Thời gian chờ giữa các lần thử (giây)** — Tăng gấp đôi sau mỗi lần thất bại.
initial_retry_delay_seconds = 15 #@param {type:"integer"}


# ===================================================================
# PROGRESS LOG SYSTEM — Theo dõi tiến trình xuyên tài khoản
# ===================================================================
PROGRESS_LOG_FILENAME = ".gdrive_turbo_copy_progress.json"

class ProgressLog:
    """Lưu và đọc tiến trình copy xuyên tài khoản."""

    def __init__(self, dest_folder_id):
        self.dest_folder_id = dest_folder_id
        self.log_file_id = None
        self.copied_ids = set()
        self.total_bytes = 0
        self.total_files = 0
        self.files_this_session = 0
        self.bytes_this_session = 0
        self.sessions = []
        self._save_counter = 0
        self._auto_save_interval = 100  # Auto-save mỗi 100 file

    def load_from_drive(self):
        """Đọc progress log từ thư mục đích trên Drive."""
        try:
            response = drive_service.files().list(
                q=f"name='{PROGRESS_LOG_FILENAME}' and '{self.dest_folder_id}' in parents and trashed=false",
                fields="files(id, name)", pageSize=1).execute()
            files = response.get('files', [])
            if not files:
                print("  Khong tim thay progress log. Bat dau moi.")
                return

            self.log_file_id = files[0]['id']
            request = drive_service.files().get_media(fileId=self.log_file_id)
            buffer = io.BytesIO()
            downloader = MediaIoBaseDownload(buffer, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()

            buffer.seek(0)
            data = json.loads(buffer.read().decode('utf-8'))
            self.copied_ids = set(data.get('copied_ids', []))
            self.total_bytes = data.get('total_bytes', 0)
            self.total_files = data.get('total_files', 0)
            self.sessions = data.get('sessions', [])

            print(f"  Da doc progress log: {self.total_files} file ({self._format_size(self.total_bytes)}) da copy truoc do.")
            if self.sessions:
                last = self.sessions[-1]
                print(f"  Phien cuoi: {last.get('account', '?')} luc {last.get('ended', '?')}")
        except Exception as e:
            print(f"  Canh bao: Khong the doc progress log: {e}. Bat dau moi.")

    def save_to_drive(self):
        """Luu progress log vao thu muc dich tren Drive."""
        data = {
            'version': 1,
            'total_files': self.total_files,
            'total_bytes': self.total_bytes,
            'copied_ids': list(self.copied_ids),
            'sessions': self.sessions
        }
        content = json.dumps(data, ensure_ascii=False)
        media = MediaIoBaseUpload(
            io.BytesIO(content.encode('utf-8')),
            mimetype='application/json', resumable=True)

        try:
            if self.log_file_id:
                drive_service.files().update(
                    fileId=self.log_file_id, media_body=media).execute()
            else:
                metadata = {
                    'name': PROGRESS_LOG_FILENAME,
                    'parents': [self.dest_folder_id],
                    'mimeType': 'application/json'
                }
                result = drive_service.files().create(
                    body=metadata, media_body=media, fields='id').execute()
                self.log_file_id = result['id']
        except Exception as e:
            print(f"  Canh bao: Khong the luu progress log: {e}")

    def delete_from_drive(self):
        """Xoa progress log khi da copy xong toan bo."""
        if self.log_file_id:
            try:
                drive_service.files().delete(fileId=self.log_file_id).execute()
                print("  Da xoa progress log (copy hoan tat).")
            except Exception as e:
                print(f"  Canh bao: Khong the xoa progress log: {e}")

    def is_copied(self, file_id):
        """Kiem tra file da duoc copy chua."""
        return file_id in self.copied_ids

    def mark_copied(self, file_id, file_size=0):
        """Danh dau file da copy thanh cong."""
        if file_id not in self.copied_ids:
            self.copied_ids.add(file_id)
            self.total_files += 1
            self.files_this_session += 1
            if file_size and file_size > 0:
                self.total_bytes += file_size
                self.bytes_this_session += file_size

            # Auto-save moi N file
            self._save_counter += 1
            if self._save_counter >= self._auto_save_interval:
                self._save_counter = 0
                self.save_to_drive()

    def start_session(self):
        """Ghi nhan bat dau phien moi."""
        self.session_start = datetime.now(timezone.utc).isoformat()
        self.files_this_session = 0
        self.bytes_this_session = 0

    def end_session(self):
        """Ghi nhan ket thuc phien va luu."""
        self.sessions.append({
            'account': current_account,
            'started': self.session_start,
            'ended': datetime.now(timezone.utc).isoformat(),
            'files_copied': self.files_this_session,
            'bytes_copied': self.bytes_this_session
        })
        self.save_to_drive()

    def _format_size(self, size_bytes):
        if size_bytes < 1024: return f"{size_bytes} B"
        elif size_bytes < 1024**2: return f"{size_bytes/1024:.1f} KB"
        elif size_bytes < 1024**3: return f"{size_bytes/1024**2:.1f} MB"
        else: return f"{size_bytes/1024**3:.2f} GB"

    def print_summary(self):
        print("\n" + "=" * 55)
        print("  PROGRESS LOG - TONG KET TIEN TRINH")
        print("=" * 55)
        print(f"  Phien nay:     {self.files_this_session} file ({self._format_size(self.bytes_this_session)})")
        print(f"  Tong cong:     {self.total_files} file ({self._format_size(self.total_bytes)})")
        print(f"  Tai khoan:     {current_account}")
        if len(self.sessions) > 1:
            print(f"\n  Lich su cac phien:")
            for i, s in enumerate(self.sessions):
                print(f"    [{i+1}] {s.get('account','?')}: {s.get('files_copied',0)} file ({self._format_size(s.get('bytes_copied',0))})")
        print("=" * 55)


# ===================================================================
# QUOTA TRACKER — Theo dõi giới hạn 750GB/ngày
# ===================================================================
class QuotaTracker:
    """Theo doi dung luong da copy de khong vuot gioi han 750GB/ngay."""

    def __init__(self, limit_gb):
        self.limit_bytes = int(limit_gb * 1024 * 1024 * 1024) if limit_gb > 0 else 0
        self.enabled = limit_gb > 0
        self.bytes_copied = 0
        self.files_copied = 0
        self.files_skipped_quota = 0
        self.bytes_skipped_quota = 0
        self.limit_reached = False

    def can_copy(self, file_size_bytes):
        if not self.enabled: return True
        if self.limit_reached: return False
        if file_size_bytes is None or file_size_bytes == 0: return True
        if self.bytes_copied + file_size_bytes > self.limit_bytes:
            self.limit_reached = True
            return False
        return True

    def record_copy(self, file_size_bytes):
        self.files_copied += 1
        if file_size_bytes and file_size_bytes > 0:
            self.bytes_copied += file_size_bytes

    def record_skip(self, file_size_bytes):
        self.files_skipped_quota += 1
        if file_size_bytes and file_size_bytes > 0:
            self.bytes_skipped_quota += file_size_bytes

    def _format_size(self, size_bytes):
        if size_bytes < 1024: return f"{size_bytes} B"
        elif size_bytes < 1024**2: return f"{size_bytes/1024:.1f} KB"
        elif size_bytes < 1024**3: return f"{size_bytes/1024**2:.1f} MB"
        else: return f"{size_bytes/1024**3:.2f} GB"

    def print_summary(self):
        print("\n" + "=" * 55)
        print("  QUOTA TRACKER - BAO CAO QUOTA")
        print("=" * 55)
        print(f"  Da copy:   {self.files_copied} file ({self._format_size(self.bytes_copied)})")
        if self.enabled:
            remaining = max(0, self.limit_bytes - self.bytes_copied)
            usage_pct = (self.bytes_copied / self.limit_bytes * 100) if self.limit_bytes > 0 else 0
            print(f"  Quota con: {self._format_size(remaining)} / {self._format_size(self.limit_bytes)} ({usage_pct:.1f}%)")
        if self.files_skipped_quota > 0:
            print(f"  Bo qua:    {self.files_skipped_quota} file ({self._format_size(self.bytes_skipped_quota)}) do dat quota.")
        if self.limit_reached:
            print(f"\n  >>> DA DAT GIOI HAN QUOTA {daily_copy_limit_gb}GB!")
            print(f"  >>> Chuyen sang tai khoan khac hoac chay lai vao ngay mai.")
        print("=" * 55)


# --- Khoi tao tracker ---
quota_tracker = QuotaTracker(daily_copy_limit_gb)
progress_log = None  # Khoi tao sau khi co dest_folder_id


# ===================================================================
# HELPER FUNCTIONS
# ===================================================================
def get_existing_items(folder_id):
    """Lay tat ca ten file va thu muc trong thu muc cho viec tra cuu nhanh."""
    existing_items = {}
    page_token = None
    query = f"'{folder_id}' in parents and trashed = false"
    while True:
        try:
            response = drive_service.files().list(
                q=query,
                fields="nextPageToken, files(id, name, mimeType)",
                pageSize=1000,
                pageToken=page_token
            ).execute()
            for item in response.get('files', []):
                if item['name'] not in existing_items:
                    existing_items[item['name']] = {'id': item['id'], 'mimeType': item['mimeType']}
            page_token = response.get('nextPageToken', None)
            if page_token is None: break
        except Exception as e:
            print(f"  - Canh bao: Khong the liet ke muc hien co trong thu muc {folder_id}: {e}")
            break
    return existing_items


def filter_files_by_quota_and_log(files_list):
    """Loc danh sach file theo quota va progress log. Tra ve (files_ok, skipped_quota, skipped_log)."""
    files_ok = []
    skipped_quota = 0
    skipped_log = 0

    for f in files_list:
        file_size = int(f.get('size', 0)) if f.get('size') else 0

        # Check progress log truoc (nhanh hon)
        if progress_log and progress_log.is_copied(f['id']):
            skipped_log += 1
            continue

        # Check quota
        if not quota_tracker.can_copy(file_size):
            quota_tracker.record_skip(file_size)
            skipped_quota += 1
            continue

        # Pre-reserve quota
        quota_tracker.record_copy(file_size)
        files_ok.append(f)

    return files_ok, skipped_quota, skipped_log


def execute_batch_in_chunks(items, build_request_func):
    """Thuc thi danh sach muc theo nhom voi logic thu lai va exponential backoff."""
    batch_size_limit = 100
    num_items = len(items)
    if num_items == 0: return

    num_batches = math.ceil(num_items / batch_size_limit)
    progress_bar = tqdm(total=num_items, desc="Dang xu ly hang loat")

    for i in range(num_batches):
        start_index = i * batch_size_limit
        end_index = start_index + batch_size_limit
        chunk = items[start_index:end_index]

        retry_delay = initial_retry_delay_seconds
        for attempt in range(max_retries_per_chunk):
            batch = drive_service.new_batch_http_request()
            for item in chunk:
                build_request_func(batch, item)

            try:
                if len(batch._requests) > 0:
                    batch.execute()
                progress_bar.update(len(chunk))
                break
            except Exception as e:
                print(f"      - Lan thu {attempt + 1}/{max_retries_per_chunk} that bai: {e}")
                if attempt + 1 == max_retries_per_chunk:
                    print(f"      - Da het so lan thu. Bo qua nhom {len(chunk)} muc nay.")
                    progress_bar.update(len(chunk))
                    break
                print(f"      - Thu lai sau {retry_delay} giay...")
                time.sleep(retry_delay)
                retry_delay *= 2
    progress_bar.close()


# ===================================================================
# COPY FOLDER CONTENTS (ĐỆ QUY)
# ===================================================================
def copy_folder_contents(source_folder_id, destination_folder_id, source_folder_name):
    """Sao chep noi dung thu muc nguon mot cach de quy va hieu qua."""
    if quota_tracker.limit_reached:
        print(f"\n  Bo qua thu muc '{source_folder_name}' - da dat gioi han quota.")
        return

    print(f"\n  Dang xu ly thu muc con: {source_folder_name}")

    # List items in source folder - bao gom 'size'
    source_files, source_folders = [], []
    page_token = None
    while True:
        try:
            response = drive_service.files().list(
                q=f"'{source_folder_id}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType, size)", pageSize=1000, pageToken=page_token,
                supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            for item in response.get('files', []):
                (source_folders if item['mimeType'] == 'application/vnd.google-apps.folder' else source_files).append(item)
            page_token = response.get('nextPageToken', None)
            if page_token is None: break
        except Exception as e:
            print(f"  - Loi liet ke noi dung nguon '{source_folder_name}': {e}. Bo qua thu muc nay.")
            return

    if copy_scope == "Folders Only":
        source_files = []

    existing_dest_items = get_existing_items(destination_folder_id) if skip_existing_items else {}
    files_to_batch = [f for f in source_files if f['name'] not in existing_dest_items]
    folders_to_batch = [f for f in source_folders if f['name'] not in existing_dest_items]

    # Loc file theo quota va progress log
    files_to_batch, skipped_q, skipped_l = filter_files_by_quota_and_log(files_to_batch)
    if skipped_q > 0:
        print(f"    Bo qua {skipped_q} file do dat quota {daily_copy_limit_gb}GB/ngay.")
    if skipped_l > 0:
        print(f"    Bo qua {skipped_l} file da copy (theo progress log).")
    print(f"    Se copy {len(files_to_batch)} file moi va {len(folders_to_batch)} thu muc moi.")

    if files_to_batch:
        copy_errors = []
        def file_copy_callback(request_id, response, exception):
            if exception:
                copy_errors.append(f"{request_id}: {exception}")
            elif progress_log:
                # Ghi nhan file da copy thanh cong
                file_size = next((int(f.get('size', 0) or 0) for f in files_to_batch if f['id'] == request_id), 0)
                progress_log.mark_copied(request_id, file_size)

        def build_file_copy_request(batch, file_item):
            batch.add(
                drive_service.files().copy(
                    fileId=file_item['id'],
                    body={'name': file_item['name'], 'parents': [destination_folder_id]},
                    supportsAllDrives=True),
                request_id=file_item['id'],
                callback=file_copy_callback)
        execute_batch_in_chunks(files_to_batch, build_file_copy_request)
        if copy_errors:
            print(f"    Canh bao: {len(copy_errors)} file copy that bai.")
            for err in copy_errors[:3]:
                print(f"      - {err}")

    newly_created_folders = {}
    if folders_to_batch:
        def folder_creation_callback(request_id, response, exception):
            if exception:
                print(f"    Loi tao thu muc (source ID: {request_id}): {exception}")
            else:
                newly_created_folders[request_id] = response
        def build_folder_create_request(batch, folder_item):
            metadata = {'name': folder_item['name'], 'mimeType': 'application/vnd.google-apps.folder', 'parents': [destination_folder_id]}
            batch.add(drive_service.files().create(body=metadata, fields='id, name'), request_id=folder_item['id'], callback=folder_creation_callback)
        execute_batch_in_chunks(folders_to_batch, build_folder_create_request)

    for source_folder in source_folders:
        dest_folder_id_for_recursion = None
        if source_folder['id'] in newly_created_folders:
            dest_folder_id_for_recursion = newly_created_folders[source_folder['id']]['id']
        elif skip_existing_items and source_folder['name'] in existing_dest_items:
            dest_folder_id_for_recursion = existing_dest_items[source_folder['name']]['id']
        if dest_folder_id_for_recursion:
            copy_folder_contents(source_folder['id'], dest_folder_id_for_recursion, source_folder['name'])


# ===================================================================
# MAIN
# ===================================================================
def main():
    global progress_log

    print("--- Bat Dau Sao Chep Tu 'Duoc Chia Se Voi Toi' ---")
    print(f"  Tai khoan: {current_account}")
    if quota_tracker.enabled:
        print(f"  Gioi han quota: {daily_copy_limit_gb}GB/ngay")

    # Step 1: Determine the destination folder ID
    if destination_folder_name:
        print(f"\nDang tim thu muc dich: '{destination_folder_name}'...")
        try:
            safe_dest_name = destination_folder_name.replace("'", "\\'")
            response = drive_service.files().list(
                q=f"name='{safe_dest_name}' and mimeType='application/vnd.google-apps.folder' and 'root' in parents and trashed=false",
                fields='files(id, name)').execute()
            if not response['files']:
                print(f"  Khong tim thay thu muc dich '{destination_folder_name}' trong 'My Drive'. Vui long tao truoc.")
                return
            dest_root_id = response['files'][0]['id']
            print(f"  Da dat dich la '{destination_folder_name}' (ID: {dest_root_id})")
        except Exception as e:
            print(f"  Loi tim thu muc dich: {e}")
            return
    else:
        dest_root_id = drive_service.files().get(fileId='root', fields='id').execute()['id']
        print("  Da dat dich la thu muc goc 'My Drive'.")

    # Step 1.5: Khoi tao va doc progress log
    print("\nDang doc progress log...")
    progress_log = ProgressLog(dest_root_id)
    progress_log.load_from_drive()
    progress_log.start_session()

    # Step 2: Get the list of items to process
    items_to_process = []
    if specific_item_name:
        print(f"\nDang tim trong 'Duoc chia se voi toi' muc ten: '{specific_item_name}'...")
        safe_item_name = specific_item_name.replace("'", "\\'")
        query = f"name = '{safe_item_name}' and sharedWithMe and trashed = false"
        try:
            response = drive_service.files().list(q=query, fields="files(id, name, mimeType, size)", pageSize=10,
                supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            found_items = response.get('files', [])
            if not found_items:
                print(f"  Khong tim thay muc ten '{specific_item_name}'.")
                return
            if len(found_items) > 1:
                print(f"  Canh bao: Tim thay {len(found_items)} muc ten '{specific_item_name}'. Xu ly muc dau tien.")
            items_to_process.append(found_items[0])
            print(f"  Da tim thay muc cu the de xu ly.")
        except Exception as e:
            print(f"  Loi khi tim muc cu the: {e}")
            return
    else:
        print("\nDang liet ke tat ca muc trong 'Duoc chia se voi toi'. Vui long doi...")
        page_token = None
        while True:
            try:
                response = drive_service.files().list(q="sharedWithMe and trashed = false",
                    fields="nextPageToken, files(id, name, mimeType, size)", pageSize=1000, pageToken=page_token,
                    supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
                items_to_process.extend(response.get('files', []))
                page_token = response.get('nextPageToken', None)
                if page_token is None: break
            except Exception as e:
                print(f"  Loi khi liet ke tat ca muc duoc chia se: {e}")
                return

    # Step 3: Filter
    shared_files = [item for item in items_to_process if item['mimeType'] != 'application/vnd.google-apps.folder']
    shared_folders = [item for item in items_to_process if item['mimeType'] == 'application/vnd.google-apps.folder']

    if copy_scope == "Files Only": shared_folders = []
    elif copy_scope == "Folders Only": shared_files = []

    total_size = sum(int(f.get('size', 0)) for f in shared_files if f.get('size'))
    print(f"\nTim thay {len(shared_files)} file ({quota_tracker._format_size(total_size)}) va {len(shared_folders)} thu muc.")

    existing_top_level_items = get_existing_items(dest_root_id) if skip_existing_items else {}
    files_to_copy = [f for f in shared_files if f['name'] not in existing_top_level_items]
    folders_to_copy = [f for f in shared_folders if f['name'] not in existing_top_level_items]

    copy_size = sum(int(f.get('size', 0)) for f in files_to_copy if f.get('size'))
    print(f"Se sao chep {len(files_to_copy)} file moi ({quota_tracker._format_size(copy_size)}) va {len(folders_to_copy)} thu muc moi.")

    if quota_tracker.enabled and copy_size > quota_tracker.limit_bytes:
        print(f"  Tong dung luong ({quota_tracker._format_size(copy_size)}) vuot quota ({daily_copy_limit_gb}GB).")
        print(f"  Se copy toi da {daily_copy_limit_gb}GB roi dung. Chuyen tai khoan hoac chay lai ngay mai.")

    # Step 4: Copy files
    files_to_copy, skipped_q, skipped_l = filter_files_by_quota_and_log(files_to_copy)
    if skipped_q > 0:
        print(f"  Bo qua {skipped_q} file cap tren cung do dat quota.")
    if skipped_l > 0:
        print(f"  Bo qua {skipped_l} file cap tren cung da copy (theo progress log).")

    if files_to_copy:
        print("\nDang sao chep file cap tren cung...")
        copy_errors = []
        def top_copy_callback(request_id, response, exception):
            if exception:
                copy_errors.append(f"{request_id}: {exception}")
            elif progress_log:
                file_size = next((int(f.get('size', 0) or 0) for f in files_to_copy if f['id'] == request_id), 0)
                progress_log.mark_copied(request_id, file_size)

        def build_top_file_copy_request(batch, file_item):
            batch.add(
                drive_service.files().copy(
                    fileId=file_item['id'],
                    body={'name': file_item['name'], 'parents': [dest_root_id]},
                    supportsAllDrives=True),
                request_id=file_item['id'],
                callback=top_copy_callback)
        execute_batch_in_chunks(files_to_copy, build_top_file_copy_request)
        if copy_errors:
            print(f"  Canh bao: {len(copy_errors)} file copy that bai.")

    # Step 5: Create and recurse folders
    newly_created_top_folders = {}
    if folders_to_copy:
        print("\nDang tao thu muc cap tren cung...")
        def top_folder_callback(request_id, response, exception):
            if exception:
                print(f"    Loi tao thu muc cap tren (source ID: {request_id}): {exception}")
            else:
                newly_created_top_folders[request_id] = response
        def build_top_folder_create_request(batch, folder_item):
            metadata = {'name': folder_item['name'], 'mimeType': 'application/vnd.google-apps.folder', 'parents': [dest_root_id]}
            batch.add(drive_service.files().create(body=metadata, fields='id, name'), request_id=folder_item['id'], callback=top_folder_callback)
        execute_batch_in_chunks(folders_to_copy, build_top_folder_create_request)

    for folder in folders_to_copy:
        if folder['id'] in newly_created_top_folders:
            dest_folder_id = newly_created_top_folders[folder['id']]['id']
            copy_folder_contents(folder['id'], dest_folder_id, folder['name'])

    # De quy vao thu muc da ton tai
    if skip_existing_items:
        for folder in shared_folders:
            if folder['name'] in existing_top_level_items and folder not in folders_to_copy:
                existing_folder_id = existing_top_level_items[folder['name']]['id']
                copy_folder_contents(folder['id'], existing_folder_id, folder['name'])

    # Step 6: Luu progress log va in bao cao
    progress_log.end_session()
    progress_log.print_summary()
    quota_tracker.print_summary()

    if quota_tracker.limit_reached:
        print("\n" + "=" * 55)
        print("  HUONG DAN TIEP TUC")
        print("=" * 55)
        print("  1. Chuyen sang tai khoan Google khac (co quyen truy cap file nguon)")
        print("  2. Chay lai notebook nay voi CUNG thong so:")
        print(f"     - destination_folder_name = '{destination_folder_name}'")
        print(f"     - skip_existing_items = True")
        print("  3. Progress log se tu dong doc va tiep tuc tu cho da dung")
        print("=" * 55)
    else:
        # Hoan tat - xoa progress log vi khong con can
        progress_log.delete_from_drive()
        print("\n\n  HOAN TAT QUA TRINH!")


# --- Run the main function ---
if __name__ == "__main__":
    main()
